<a href="https://colab.research.google.com/github/liangliang6v6/Homeworks/blob/pages/Homework6_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Task 1 (30 points):
In this task you should work with the Facebook BART model
(https://huggingface.co/docs/transformers/en/model_doc/bart) to provide text summarization
of news articles. Text summarization in Natural Language Processing (NLP) is a technique that
breaks down long texts into sentences or paragraphs, while retaining the text's meaning and
extracting important information. Pick any one dataset of your choice.
You may have to do data cleaning, preprocessing etc. Next, perform the following tasks:
1. Provide a description of the dataset you selected. Split your data into train-test set with
a (90-10) split.
2. Load the model from Hugging Face’s Transformers library and write its training script.
3. Fine tune the pre-trained model with your data and report results on your test set. You
must report the BLEU and ROUGE Scores. (See the code provided in class for more
details)
4. Analyze your results and discuss the impact of hyperparameters. Are your results
impacted by the choice of the LLM here? How?

## Answer
The XSum (Extreme Summarization) dataset is a large-scale dataset designed for abstractive text summarization, consisting of 226,711 BBC news articles paired with highly concise, one-sentence summaries. Unlike extractive summarization, XSum requires models to generate summaries that are reworded and synthesized rather than directly copied from the source text. The dataset is widely used for training and evaluating summarization models, particularly in deep learning research. It is available through the Hugging Face Datasets library and includes predefined splits for training, validation, and testing.

Available link: https://huggingface.co/datasets/EdinburghNLP/xsum

The model's performance, with a BLEU score of around 0.04 and moderate ROUGE scores, indicates challenges in producing precise and coherent summaries. It's important to note that the model was trained on a very small subset of the dataset for efficiency. This limited data size impacts the model's ability to fully learn patterns and nuances, potentially contributing to the lower scores. Adjusting hyperparameters, such as the number of training epochs, learning rate, and batch sizes, might improve performance by allowing the model to better adapt to the dataset's specifics.

Choosing BART as the model provides significant advantages due to its capability to capture context—essential for summarization. However, using a tiny dataset limits the ability to leverage BART’s full potential. Even though BART offers a strong baseline, performance is closely tied to the dataset's size and the extent of fine-tuning. By using more data or refining training strategies, the model's results could be enhanced significantly. Thus, while BART sets a solid groundwork, optimizing hyperparameters in conjunction with a small dataset presents unique challenges that influence the overall outcomes.

In [5]:
!pip install datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.2/491.2 kB 21.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 10.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.9/183.9 kB 11.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.5/143.5 kB 9.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.8/194.8 kB 15.3 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2025.3.0
    Uninstalling fsspec-2025.3.0:
      Successfully uninstalled fsspec-2025.3.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
torch 2.6.0+cu124 requires nvidia-cublas-cu12==12.4.5.8; platform_system == "Linux" and platform_machine == "x86_64", but you have nvidia-cublas-cu12 12.5.3.2 which is incompatible.
torch 2.6.0+cu124 requires nvidia-cuda-cupti-cu12==12.4.127; platform_system =

In [14]:
from datasets import load_dataset
from sklearn.model_selection import train_test_split

dataset = load_dataset("xsum")

data = dataset['train'].to_pandas()

# split data into train and test sets (90-10 split)
train_data, test_data = train_test_split(data, test_size=0.1, random_state=42)

print(f"Training set size: {len(train_data)}")
print(f"Test set size: {len(test_data)}")

Training set size: 183640
Test set size: 20405


In [6]:
import torch
from transformers import BartTokenizer, BartForConditionalGeneration, TrainingArguments
from datasets import load_dataset
from torch.utils.data import DataLoader

# use GPU for train
tokenizer = BartTokenizer.from_pretrained('facebook/bart-large-cnn')
model = BartForConditionalGeneration.from_pretrained('facebook/bart-large-cnn')

def tokenize_data(examples):
    inputs = tokenizer(examples['document'], max_length=512, truncation=True, padding="max_length")
    targets = tokenizer(examples['summary'], max_length=64, truncation=True, padding="max_length")
    return {
        'input_ids': inputs['input_ids'],
        'attention_mask': inputs['attention_mask'],
        'labels': targets['input_ids']
    }

train_tokenized = train_data.map(tokenize_data, batched=True)
test_tokenized = test_data.map(tokenize_data, batched=True)


train_tokenized.set_format(type='torch', columns=['input_ids', 'attention_mask', 'labels'])
test_tokenized.set_format(type='torch', columns=['input_ids', 'attention_mask', 'labels'])

def collate_fn(batch):
    input_ids = torch.stack([x['input_ids'] for x in batch])
    attention_mask = torch.stack([x['attention_mask'] for x in batch])
    labels = torch.stack([x['labels'] for x in batch])
    return {'input_ids': input_ids, 'attention_mask': attention_mask, 'labels': labels}

train_loader = DataLoader(train_tokenized, batch_size=2, shuffle=True, collate_fn=collate_fn)
test_loader = DataLoader(test_tokenized, batch_size=2, collate_fn=collate_fn)

training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=10,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    warmup_steps=100,
    weight_decay=0.01,
    logging_dir='./logs',
    logging_steps=10,
    eval_strategy='epoch'
)

def train_model():
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-5)
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model.to(device)
    print("Training on device:", device)

    for epoch in range(training_args.num_train_epochs):
        model.train()
        for batch in train_loader:
            optimizer.zero_grad()
            inputs = {key: val.to(device) for key, val in batch.items()}
            outputs = model(**inputs)
            loss = outputs.loss
            loss.backward()
            optimizer.step()
            print(f'Epoch {epoch + 1}, Loss: {loss.item()}')

# Train the model
train_model()

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/6.24k [00:00<?, ?B/s]

xsum.py:   0%|          | 0.00/5.76k [00:00<?, ?B/s]

The repository for xsum contains custom code which must be executed to correctly load the dataset. You can inspect the repository content at https://hf.co/datasets/xsum.
You can avoid this prompt in future by passing the argument `trust_remote_code=True`.

Do you wish to run the custom code? [y/N] y


(…)SUM-EMNLP18-Summary-Data-Original.tar.gz:   0%|          | 0.00/255M [00:00<?, ?B/s]

...

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

Map:   0%|          | 0/50 [00:00<?, ? examples/s]

Training on device: cuda
Epoch 1, Loss: 7.906005859375
Epoch 1, Loss: 7.503452301025391
Epoch 1, Loss: 7.107475280761719
Epoch 1, Loss: 6.806521892547607
Epoch 1, Loss: 6.21462345123291
Epoch 1, Loss: 7.305330276489258
Epoch 1, Loss: 6.082671165466309
Epoch 1, Loss: 5.927668571472168
Epoch 1, Loss: 6.235300064086914
Epoch 1, Loss: 6.357330322265625
Epoch 1, Loss: 5.745393753051758
Epoch 1, Loss: 5.916208267211914
Epoch 1, Loss: 4.05690336227417
Epoch 1, Loss: 5.37644100189209
Epoch 1, Loss: 5.473827838897705
Epoch 1, Loss: 5.3470683097839355
Epoch 1, Loss: 5.073696136474609
Epoch 1, Loss: 5.36064338684082
Epoch 1, Loss: 4.686415672302246
Epoch 1, Loss: 4.3750128746032715
Epoch 1, Loss: 3.9221651554107666
Epoch 1, Loss: 4.143355369567871
Epoch 1, Loss: 4.16127347946167
Epoch 1, Loss: 3.9492852687835693
Epoch 1, Loss: 3.5828075408935547
Epoch 1, Loss: 3.7738420963287354
Epoch 1, Loss: 3.353658676147461
Epoch 1, Loss: 2.824822425842285
Epoch 1, Loss: 2.929990530014038
Epoch 1, Loss: 2.782

In [11]:
!pip install rouge_score

  Preparing metadata (setup.py) ... done
  Created wheel for rouge_score: filename=rouge_score-0.1.2-py3-none-any.whl size=24934 sha256=2fea87021d656f73abd8b5bc7cdadaed6b10d80daa9ae535613f3773e563c039
  Stored in directory: /root/.cache/pip/wheels/1e/19/43/8a442dc83660ca25e163e1bd1f89919284ab0d0c1475475148
Successfully built rouge_score


In [9]:
!pip install evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.0/84.0 kB 5.8 MB/s eta 0:00:00


In [15]:
import evaluate

# Load metrics
rouge_metric = evaluate.load("rouge")
bleu_metric = evaluate.load("bleu")

def evaluate_model(model, test_loader, tokenizer):
    model.eval()
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model.to(device)

    predictions = []
    references = []

    for batch in test_loader:
        inputs = {key: val.to(device) for key, val in batch.items() if key != 'labels'}
        with torch.no_grad():
            outputs = model.generate(**inputs, max_length=64)

        predicted_summaries = [tokenizer.decode(g, skip_special_tokens=True, clean_up_tokenization_spaces=True) for g in outputs]

        references.extend([tokenizer.decode(labels, skip_special_tokens=True, clean_up_tokenization_spaces=True) for labels in batch['labels']])
        predictions.extend(predicted_summaries)

    return predictions, references

# Evaluate
predictions, references = evaluate_model(model, test_loader, tokenizer)

# Calculate BLEU
bleu_score = bleu_metric.compute(predictions=predictions, references=references)

# Calculate ROUGE
rouge_score = rouge_metric.compute(predictions=predictions, references=references)

print("BLEU Score:", bleu_score)
print("ROUGE Score:", rouge_score)

BLEU Score: {'bleu': 0.04013238870215141, 'precisions': [0.1925841846386682, 0.05360586193598149, 0.02320094376720409, 0.010830324909747292], 'brevity_penalty': 1.0, 'length_ratio': 2.2922810060711187, 'translation_length': 2643, 'reference_length': 1153}
ROUGE Score: {'rouge1': np.float64(0.26568964450524646), 'rouge2': np.float64(0.08871078806064278), 'rougeL': np.float64(0.18890856186949265), 'rougeLsum': np.float64(0.18950659718092078)}
